# 03. Geographic frame

This notebook builds the spatial units the rest of the analysis runs on: the
boundary of each of the 92 metropolitan areas, and the census blocks that fall
inside them.

Blocks are the finest unit at which the 2020 census publishes population, and
they are what travel times are measured from, so the block frame determines both
the resolution of the analysis and which parts of the country are in it. Blocks
are grouped into census tracts, which is the level at which deprivation is
published and at which results are reported.

The two inputs arrive in different coordinate systems and at different levels of
aggregation. Metropolitan area boundaries are supplied as one file per area,
each holding the municipalities that compose it in geographic coordinates. Block
geometries are supplied as a single national file in a projected system. Every
bounding box passed between them is converted into the coordinate system of the
file being read, because a box in the wrong system silently selects nothing and
returns an empty result rather than an error.

Each metropolitan area is processed separately and cached, so an interrupted run
resumes where it stopped.

Output is `Results/03_blocks.parquet`, one row per census block with its
identifier, its tract, its metropolitan area and its area in square metres,
`Results/03_metro_boundaries.gpkg`, one dissolved polygon per metropolitan area,
and a record of how many units were dropped at each step and why.

Runtime is roughly one hour, most of it reading block geometry.

## 1. Setup

In [ ]:
import sys, os, glob, zipfile, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'module is pointing at {mx.BASE}'

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio

rep = mx.Report('03_geographic_frame').start()
mx.flow_reset()

CACHE = mx.cache_dir('cache_03_blocks')
INPUTS = mx.DIRS['inputs']
print('cache directory:', CACHE)
print('working projection: EPSG:%d (metres)' % mx.CRS_METRIC)

## 2. Metropolitan area boundaries

Each area is supplied as one file holding the municipalities that compose it.
The municipalities are dissolved into a single polygon per area, which is what
every later clip is taken against.

The files also carry the official classification of each area, so the
classification used throughout the analysis is checked against the source rather
than assumed to match it.

In [ ]:
BOUND_DIR = f'{INPUTS}/metro_areas'
bfiles = sorted(glob.glob(os.path.join(BOUND_DIR, 'metro_*.gpkg')))
assert len(bfiles) == 92, f'{len(bfiles)} boundary files found, expected 92'
print(f'{len(bfiles)} metropolitan area boundary files')

metros = mx.load_metros()

parts = []
for p in bfiles:
    lyr = mx.first_layer(p)
    g = gpd.read_file(p, layer=lyr, engine='pyogrio')
    if len(g) == 0:
        raise RuntimeError(f'{os.path.basename(p)} contains no municipalities')
    g['src_file'] = os.path.basename(p)
    g['slug'] = mx._slug_name([os.path.basename(p)[:-5]]).iloc[0]
    parts.append(g)

MUNI = pd.concat(parts, ignore_index=True)
MUNI = gpd.GeoDataFrame(MUNI, geometry='geometry', crs=parts[0].crs)
print(f'{len(MUNI):,} municipalities across {MUNI.slug.nunique()} areas')
print('source projection:', MUNI.crs)
print('fields:', [c for c in MUNI.columns if c != 'geometry'])

In [ ]:
# Dissolve municipalities into one polygon per metropolitan area, in metres.
METRO = (MUNI.to_crs(epsg=mx.CRS_METRIC)
             .dissolve(by='slug', aggfunc={'CVE_MET': 'first', 'NOM_MET': 'first',
                                           'TIPO_MET': 'first', 'CVEGEO': 'size'})
             .rename(columns={'CVEGEO': 'n_municipalities'})
             .reset_index())
METRO['area_km2'] = (METRO.geometry.area / 1e6).round(1)

mx.assert_join(METRO['slug'], mx._slug_name(metros['metro_key']),
               'boundary files to crosswalk', 1.0)
METRO = METRO.merge(metros.assign(slug=mx._slug_name(metros['metro_key']).values)
                          [['slug', 'metro_key', 'metro_name', 'tipo', 'tier',
                            'conapo_pop_2020']],
                    on='slug', how='left')
assert len(METRO) == 92 and METRO['metro_key'].notna().all()

print(f'\n{len(METRO)} metropolitan areas, '
      f'{METRO.n_municipalities.sum():,} municipalities, '
      f'{METRO.area_km2.sum():,.0f} km2 total')

In [ ]:
# The classification carried by the boundary files against the one used here.
# A mismatch means the two disagree about what an area is, which has to be
# resolved before any result is reported by class.
xtab = pd.crosstab(METRO['TIPO_MET'], METRO['tipo'])
print('official classification in the boundary files against the one in use:\n')
print(xtab.to_string())

off = mx._slug_name(METRO['TIPO_MET'])
use = mx._slug_name(METRO['tipo'])
mismatch = METRO.loc[off.values != use.values,
                     ['slug', 'metro_name', 'TIPO_MET', 'tipo']]
print(f'\nareas where the two disagree: {len(mismatch)}')
if len(mismatch):
    print(mismatch.to_string(index=False))
rep.add_table('classification cross-check', xtab.reset_index(), max_rows=10)

## 3. Locating the block geometries

The national geostatistical framework is distributed as zipped shapefiles. Each
archive is opened through its own metadata to establish what it holds, which
file is which, how many features it has and in what projection, rather than
relying on the file names.

The archive holding census blocks is identified by its key field, which is a
sixteen character identifier combining state, municipality, locality, tract and
block.

In [ ]:
MGN_DIR = f'{INPUTS}/MGN/manzanas'
mgn_zips = sorted(glob.glob(os.path.join(MGN_DIR, '*.zip')))
assert mgn_zips, f'no archives in {MGN_DIR}'

rows = []
for z in mgn_zips:
    with zipfile.ZipFile(z) as zf:
        shps = [n for n in zf.namelist() if n.lower().endswith('.shp')]
    for s in shps:
        vsi = f'/vsizip/{z}/{s}'
        info = pyogrio.read_info(vsi)
        rows.append({
            'archive': os.path.basename(z),
            'shapefile': s,
            'vsi_path': vsi,
            'features': info.get('features'),
            'geometry': info.get('geometry_type'),
            'crs': str(info.get('crs')),
            'fields': list(info['fields']),
        })

MGN = pd.DataFrame(rows)
print(MGN[['archive', 'shapefile', 'features', 'geometry', 'crs']].to_string(index=False))
print()
for r in MGN.itertuples():
    print(f'{r.archive}: {len(r.fields)} fields')
    print('   ', ', '.join(r.fields))
rep.add_table('geostatistical framework contents',
              MGN.drop(columns=['fields', 'vsi_path']).assign(
                  n_fields=MGN['fields'].str.len()), max_rows=20)

In [ ]:
# The block layer is the polygon layer carrying a CVEGEO field whose values are
# sixteen characters long. Identifying it by content rather than by file name
# means a renamed or reordered distribution does not silently select the wrong
# layer. The choice is printed so it can be seen rather than inferred.

def _key_width(vsi, field='CVEGEO', n=50):
    try:
        head = pyogrio.read_dataframe(vsi, columns=[field], max_features=n,
                                      read_geometry=False)
    except Exception:
        return None
    if field not in head.columns or head.empty:
        return None
    return int(head[field].astype(str).str.len().mode().iloc[0])

MGN['key_width'] = [
    _key_width(r.vsi_path) if 'CVEGEO' in r.fields else None
    for r in MGN.itertuples()
]
print(MGN[['archive', 'shapefile', 'features', 'geometry', 'key_width']]
      .to_string(index=False))

cand = MGN[(MGN['key_width'] == 16) & MGN['geometry'].str.contains('Polygon', na=False)]
if len(cand) != 1:
    raise RuntimeError(
        'could not identify exactly one block layer; candidates:\n'
        f'{MGN[["archive", "shapefile", "features", "geometry", "key_width"]].to_string(index=False)}')

BLOCK_SRC = cand.iloc[0]
BLOCK_CRS = BLOCK_SRC['crs']
print(f"\nblock geometries: {BLOCK_SRC['archive']} / {BLOCK_SRC['shapefile']}")
print(f"   {BLOCK_SRC['features']:,} features, projection {BLOCK_CRS}")

The block file is extracted to the machine's own disk before it is read. A
shapefile stores its spatial index in a companion file, and that index is what
makes it possible to read a small window out of a large file instead of scanning
all of it. Reading the shapefile from inside the archive, or across a mounted
drive, bypasses the index, and the national block file is large enough that the
difference is between minutes and hours across 92 reads.

In [ ]:
import shutil

LOCAL = '/content/mgn_blocks'
os.makedirs(LOCAL, exist_ok=True)

stem = os.path.splitext(os.path.basename(BLOCK_SRC['shapefile']))[0]
arch = os.path.join(MGN_DIR, BLOCK_SRC['archive'])
local_shp = os.path.join(LOCAL, stem + '.shp')

if not os.path.exists(local_shp):
    with zipfile.ZipFile(arch) as zf:
        members = [n for n in zf.namelist()
                   if os.path.splitext(os.path.basename(n))[0] == stem]
        print(f'extracting {len(members)} files to {LOCAL}')
        for n in members:
            with zf.open(n) as fh, open(os.path.join(LOCAL, os.path.basename(n)), 'wb') as out:
                shutil.copyfileobj(fh, out)

have = sorted(os.path.splitext(f)[1].lower() for f in os.listdir(LOCAL))
print('extracted components:', ', '.join(have))
print('spatial index present:', any(e in have for e in ('.sbn', '.qix')))

BLOCK_PATH = local_shp
info = pyogrio.read_info(BLOCK_PATH)
BLOCK_CRS = str(info['crs'])
print(f"reading from {BLOCK_PATH}")
print(f"   {info['features']:,} features, projection {BLOCK_CRS}")

## 4. Reading blocks inside each metropolitan area

Blocks are selected in two stages. A bounding box drawn around the metropolitan
area, converted into the projection of the block file, narrows the read to a
manageable number of features. Those features are then kept only where their
centre falls inside the area itself, so a block on the far side of the bounding
box is not included.

A metropolitan area that returns no blocks stops the run. An empty result here
would otherwise propagate as an area with no population rather than as an error.

In [ ]:
def blocks_for_metro(geom_metric, pad_m=2000.0):
    """Census blocks whose centre lies inside one metropolitan area."""
    from shapely.geometry import box as _box

    bx = np.array(geom_metric.bounds) + np.array([-pad_m, -pad_m, pad_m, pad_m])
    bb = (gpd.GeoSeries([_box(*bx)], crs=f'EPSG:{mx.CRS_METRIC}')
            .to_crs(BLOCK_CRS).total_bounds)

    raw = gpd.read_file(BLOCK_PATH, columns=['CVEGEO'],
                        engine='pyogrio', bbox=tuple(bb))
    if len(raw) == 0:
        raise RuntimeError('bounding box selected no blocks')

    raw = raw.to_crs(epsg=mx.CRS_METRIC)
    inside = raw[raw.geometry.representative_point().within(geom_metric)].copy()
    inside['area_m2'] = inside.geometry.area
    return raw, inside

In [ ]:
frames = []
scan = []
for i, r in enumerate(METRO.itertuples(), 1):
    cpath = os.path.join(CACHE, f'{r.slug}.parquet')
    if os.path.exists(cpath):
        df = pd.read_parquet(cpath)
        frames.append(df)
        scan.append({'slug': r.slug, 'in_box': np.nan, 'kept': len(df)})
        continue

    with mx.stage(f'[{i:>2}/92] {r.slug}'):
        raw, inside = blocks_for_metro(r.geometry)
        if len(inside) == 0:
            raise RuntimeError(
                f'{r.slug}: {len(raw):,} blocks in the bounding box but none '
                'inside the area boundary')
        df = pd.DataFrame({
            'CVEGEO_MZA': inside['CVEGEO'].astype(str).values,
            'slug': r.slug,
            'metro_key': r.metro_key,
            'area_m2': inside['area_m2'].round(1).values,
        })
        df.to_parquet(cpath, index=False)
    frames.append(df)
    scan.append({'slug': r.slug, 'in_box': len(raw), 'kept': len(inside)})

BLOCKS = pd.concat(frames, ignore_index=True)
SCAN = pd.DataFrame(scan)
print(f'\n{len(BLOCKS):,} blocks across {BLOCKS.slug.nunique()} areas')

## 5. Deriving the tract identifier

A block identifier is the tract identifier followed by a three digit block
number, so the tract is the first thirteen characters. Rather than slicing and
trusting the result, the identifier is taken apart into its components and
reassembled, which fails loudly on any key of the wrong length.

In [ ]:
k = BLOCKS['CVEGEO_MZA'].astype(str)
bad = k[k.str.len() != 16]
if len(bad):
    raise ValueError(f'{len(bad):,} block keys are not 16 characters: '
                     f'{bad.head().tolist()}')

BLOCKS['CVEGEO'] = mx.build_cvegeo(k.str[:2], k.str[2:5], k.str[5:9], k.str[9:13])
BLOCKS['CVE_ENT'] = k.str[:2]
BLOCKS['MZA'] = k.str[13:16]

rebuilt = BLOCKS['CVEGEO'] + BLOCKS['MZA']
assert (rebuilt == k).all(), 'reassembled block keys do not match the originals'
print('block keys decompose and reassemble cleanly')

BLOCKS['macro_region'] = mx.macro_region(BLOCKS['CVE_ENT'])
print(f"\n{BLOCKS['CVEGEO'].nunique():,} tracts, {len(BLOCKS):,} blocks")

## 6. Attrition

In [ ]:
mx.flow('blocks inside a metropolitan boundary', len(BLOCKS),
        'centre within the dissolved area')
mx.flow('blocks with a valid 16-character key', int((k.str.len() == 16).sum()),
        'key structure verified')
mx.flow('blocks with positive area', int((BLOCKS.area_m2 > 0).sum()),
        'degenerate geometry excluded')

FRAME = BLOCKS[BLOCKS.area_m2 > 0].copy()
mx.flow('blocks in the final frame', len(FRAME), 'carried forward')
mx.flow('tracts in the final frame', FRAME['CVEGEO'].nunique(), 'distinct tracts')

print()
print(mx.flow_table().to_string(index=False))
rep.add_table('attrition', mx.flow_table(), max_rows=20)

## 7. Checks

In [ ]:
print('--- checks ---\n')

# Fails if a block identifier appears twice, which would double count its
# population and its travel time.
dups = int(FRAME['CVEGEO_MZA'].duplicated().sum())
print(f'1. duplicate block identifiers                 {dups}')
assert dups == 0, FRAME.loc[FRAME.CVEGEO_MZA.duplicated(), 'CVEGEO_MZA'].head().tolist()

# Fails if a block was assigned to more than one metropolitan area, which can
# happen where two areas adjoin and both claim the boundary.
multi = FRAME.groupby('CVEGEO_MZA')['slug'].nunique()
n_multi = int((multi > 1).sum())
print(f'2. blocks claimed by two areas                 {n_multi}')
assert n_multi == 0, multi[multi > 1].head().to_dict()

# Fails if any area contributed no blocks.
empty = sorted(set(METRO['slug']) - set(FRAME['slug']))
print(f'3. areas contributing no blocks                {len(empty)}')
assert not empty, empty

# Fails if a tract straddles two metropolitan areas, which would make tract
# level results ambiguous.
tmulti = FRAME.groupby('CVEGEO')['slug'].nunique()
n_tmulti = int((tmulti > 1).sum())
print(f'4. tracts spanning two areas                   {n_tmulti}')
if n_tmulti:
    print('   ', tmulti[tmulti > 1].head(10).to_dict())

# A proportion of blocks inside the bounding box but outside the boundary is
# expected, since the box is rectangular and the areas are not. A value near
# zero or near one would indicate the box or the containment test is wrong.
done = SCAN.dropna(subset=['in_box'])
if len(done):
    share = done['kept'].sum() / done['in_box'].sum()
    print(f'5. blocks kept from the bounding box reads     {share:.1%} '
          f'({done["kept"].sum():,} of {done["in_box"].sum():,})')

print(f'\n6. total block area                            '
      f'{FRAME.area_m2.sum() / 1e6:,.0f} km2 '
      f'against {METRO.area_km2.sum():,.0f} km2 of metropolitan area')
print(f'7. median block size                           {FRAME.area_m2.median():,.0f} m2')

## 8. Results

In [ ]:
by_metro = (FRAME.groupby(['slug'])
              .agg(blocks=('CVEGEO_MZA', 'size'),
                   tracts=('CVEGEO', 'nunique'),
                   block_km2=('area_m2', lambda s: round(s.sum() / 1e6, 1)))
              .reset_index()
              .merge(METRO[['slug', 'metro_name', 'tipo', 'tier', 'area_km2',
                            'n_municipalities', 'conapo_pop_2020']], on='slug'))

print('--- per-area frame, most blocks first ---\n')
print(by_metro.sort_values('blocks', ascending=False)
             [['slug', 'metro_name', 'tipo', 'n_municipalities', 'tracts',
               'blocks', 'block_km2', 'area_km2']]
             .to_string(index=False))

print('\n--- by official classification ---\n')
print(by_metro.groupby('tipo')
        .agg(areas=('slug', 'size'), tracts=('tracts', 'sum'),
             blocks=('blocks', 'sum'), block_km2=('block_km2', 'sum'))
        .reindex(mx.TIPO_ORDER).to_string())

print('\n--- by macro region ---\n')
print(FRAME.groupby('macro_region')
        .agg(blocks=('CVEGEO_MZA', 'size'), tracts=('CVEGEO', 'nunique'),
             areas=('slug', 'nunique')).to_string())

rep.add_table('per-area frame', by_metro, max_rows=92)

## 9. Write results

In [ ]:
out = os.path.join(mx.DIRS['results'], '03_blocks.parquet')
FRAME[['CVEGEO_MZA', 'CVEGEO', 'CVE_ENT', 'MZA', 'slug', 'metro_key',
       'macro_region', 'area_m2']].to_parquet(out, index=False)
print('written:', out, f'({os.path.getsize(out) / 1e6:.1f} MB)')

outb = os.path.join(mx.DIRS['results'], '03_metro_boundaries.gpkg')
(METRO[['slug', 'metro_key', 'metro_name', 'tipo', 'tier', 'CVE_MET',
        'TIPO_MET', 'n_municipalities', 'area_km2', 'conapo_pop_2020',
        'geometry']]
   .to_file(outb, layer='metro_boundaries', driver='GPKG'))
print('written:', outb, f'({os.path.getsize(outb) / 1e6:.1f} MB)')

mx.save_flow(os.path.join(mx.DIRS['logs'], '03_sample_flow.csv'))

rep.note(f'{len(FRAME):,} blocks in {FRAME["CVEGEO"].nunique():,} tracts across '
         f'{FRAME["slug"].nunique()} metropolitan areas; '
         f'{FRAME.area_m2.sum() / 1e6:,.0f} km2 of block area')
rep.save()